# Cross-checkpoint noise table comparison

Compare aggregated noise tables (multi5, 5 runs × 200 samples) across 3 checkpoints:
1. `uqat_cycle4b_repro` — cycle4b baseline (~53% chip acc)
2. `uqat_tmp02_refine_ndis32_0_78` — refine noise CSV NAT (~33% chip acc)
3. `uqat_observed_npz_tmp02_refine_ndis32_0_78` — observed-noise NAT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys

CIM_DIR = '/root/project/CIM'
NT_BASE = os.path.join(CIM_DIR, 'noise/noise_df/B2_chip_inference/N32')

CKPTS = {
    'cycle4b': 'uqat_cycle4b_repro',
    'refine': 'uqat_tmp02_refine_ndis32_0_78',
    'observed': 'uqat_observed_npz_tmp02_refine_ndis32_0_78',
}

# Load all 3 noise tables
tables = {}
for label, ckpt in CKPTS.items():
    path = os.path.join(NT_BASE, ckpt, 'aggregated_noise_table__multi5.npz')
    nt = np.load(path)
    tables[label] = {
        'probs': nt['probs'],
        'ref_centers': nt['ref_bin_centers'],
        'noise_centers': nt['noise_bin_centers'],
        'E': nt['E'],
        'Std': nt['Std'],
        'count': nt['count'],
    }
    p = nt['probs']
    c = nt['count']
    print(f'{label:12s}: shape={p.shape}, obs={c.sum():>12,}, '
          f'ref=[{nt["ref_bin_centers"][0]:.0f}, {nt["ref_bin_centers"][-1]:.0f}], '
          f'noise=[{nt["noise_bin_centers"][0]:.0f}, {nt["noise_bin_centers"][-1]:.0f}]')

## 1. Side-by-side heatmaps per pseudo_ch

3 rows (cycle4b / refine / observed) × N pseudo_ch columns.  
Shows log10(probability), E[noise] line, and ±1σ bands.

In [ ]:
def plot_3ckpt_heatmaps(pseudo_chs, figsize=(18, 10), min_count=10):
    """Side-by-side heatmaps: 3 checkpoints (rows) × pseudo_chs (cols)."""
    labels = list(tables.keys())
    n = len(pseudo_chs)
    fig, axes = plt.subplots(3, n, figsize=figsize, squeeze=False)

    for row, label in enumerate(labels):
        t = tables[label]
        for col, pch in enumerate(pseudo_chs):
            ax = axes[row, col]
            p = t['probs'][pch].copy()
            cnt = t['count'][pch]
            e = t['E'][pch]
            s = t['Std'][pch]
            ref_c = t['ref_centers']
            noise_c = t['noise_centers']

            valid = cnt >= min_count
            p[~valid] = np.nan
            with np.errstate(divide='ignore', invalid='ignore'):
                p_log = np.log10(p + 1e-8)
            p_log[~valid] = np.nan

            im = ax.imshow(p_log, aspect='auto', origin='lower', cmap='inferno',
                           extent=[noise_c[0], noise_c[-1], ref_c[0], ref_c[-1]],
                           vmin=-4, vmax=0)

            vidx = np.where(valid)[0]
            if len(vidx) > 0:
                ax.plot(e[vidx], ref_c[vidx], 'c-', lw=1.2, alpha=0.9, label='E[noise]')
                ax.plot(e[vidx] - s[vidx], ref_c[vidx], 'c--', lw=0.6, alpha=0.6)
                ax.plot(e[vidx] + s[vidx], ref_c[vidx], 'c--', lw=0.6, alpha=0.6, label='±1σ')

            ax.axvline(0, color='white', lw=0.5, alpha=0.4)
            if row == 2:
                ax.set_xlabel('Observed noise', fontsize=9)
            if col == 0:
                ax.set_ylabel(f'{label}\nClean ref (psum)', fontsize=9)
            ax.set_title(f'pch {pch}  (n={cnt.sum():,})', fontsize=10)
            ax.legend(fontsize=6, loc='upper left')
            ax.tick_params(labelsize=7)

    fig.subplots_adjust(right=0.92, hspace=0.3, wspace=0.3)
    cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
    fig.colorbar(im, cax=cbar_ax, label='log10(prob)')
    fig.suptitle('Noise table comparison: cycle4b vs refine vs observed', fontsize=13, y=0.98)
    plt.show()

In [ ]:
# Pick interesting pseudo_chs: high-noise channels from the summary
plot_3ckpt_heatmaps([92, 107, 105, 435, 430], figsize=(22, 10))

In [ ]:
# Low-noise channels for contrast
plot_3ckpt_heatmaps([0, 50, 100, 200, 300], figsize=(22, 10))

## 2. Per-pseudo_ch E[noise] comparison

Scatter/line plots comparing mean noise bias across the 3 checkpoints for each pseudo_ch.

In [ ]:
n_pch = tables['cycle4b']['probs'].shape[0]
min_count_per_ch = 100

# Compute per-pch weighted mean E and Std for each checkpoint
stats = {}
for label, t in tables.items():
    e_means = []
    std_means = []
    total_counts = []
    for pch in range(n_pch):
        valid = t['count'][pch] >= 10
        if valid.sum() == 0:
            e_means.append(np.nan)
            std_means.append(np.nan)
            total_counts.append(0)
            continue
        # Weighted mean by count
        c = t['count'][pch][valid].astype(float)
        e = t['E'][pch][valid]
        s = t['Std'][pch][valid]
        w = c / c.sum()
        e_means.append((e * w).sum())
        std_means.append((s * w).sum())
        total_counts.append(int(c.sum()))
    stats[label] = {
        'E_mean': np.array(e_means),
        'Std_mean': np.array(std_means),
        'total_count': np.array(total_counts),
    }

# Scatter: E_mean for each pair of checkpoints
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pairs = [('cycle4b', 'refine'), ('cycle4b', 'observed'), ('refine', 'observed')]
for ax, (l1, l2) in zip(axes, pairs):
    e1 = stats[l1]['E_mean']
    e2 = stats[l2]['E_mean']
    mask = ~(np.isnan(e1) | np.isnan(e2))
    ax.scatter(e1[mask], e2[mask], s=8, alpha=0.5)
    lims = [min(np.nanmin(e1), np.nanmin(e2)), max(np.nanmax(e1), np.nanmax(e2))]
    ax.plot(lims, lims, 'r--', lw=0.8, alpha=0.5)
    ax.set_xlabel(f'{l1} E[noise]')
    ax.set_ylabel(f'{l2} E[noise]')
    ax.set_title(f'{l1} vs {l2} ({mask.sum()} pch)')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
fig.suptitle('Per-pseudo_ch mean noise bias comparison', fontsize=13)
fig.tight_layout()
plt.show()

## 3. E[noise] difference: refine − cycle4b, observed − cycle4b

Which pseudo_chs have the largest noise bias difference between checkpoints?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

e_c4b = stats['cycle4b']['E_mean']
e_ref = stats['refine']['E_mean']
e_obs = stats['observed']['E_mean']

for ax, (e_other, lbl) in zip(axes, [(e_ref, 'refine − cycle4b'), (e_obs, 'observed − cycle4b')]):
    diff = e_other - e_c4b
    mask = ~np.isnan(diff)
    pch_idx = np.arange(n_pch)[mask]
    ax.bar(pch_idx, diff[mask], width=1.0, alpha=0.7)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('pseudo_ch')
    ax.set_ylabel('ΔE[noise]')
    ax.set_title(lbl)
    ax.grid(True, alpha=0.3, axis='y')
    # Annotate top 5 outliers
    abs_diff = np.abs(diff[mask])
    top5 = np.argsort(abs_diff)[-5:]
    for idx in top5:
        ax.annotate(f'{pch_idx[idx]}', (pch_idx[idx], diff[mask][idx]),
                    fontsize=7, ha='center', va='bottom' if diff[mask][idx] > 0 else 'top')

fig.suptitle('Noise bias difference relative to cycle4b', fontsize=13)
fig.tight_layout()
plt.show()

# Print top 10 biggest differences
for lbl, e_other in [('refine', e_ref), ('observed', e_obs)]:
    diff = e_other - e_c4b
    mask = ~np.isnan(diff)
    order = np.argsort(np.abs(diff[mask]))[::-1]
    pch_idx = np.arange(n_pch)[mask]
    print(f'\nTop 10 |ΔE| for {lbl} − cycle4b:')
    print(f'{"pch":>5s} {"cycle4b":>10s} {lbl:>10s} {"delta":>10s}')
    for i in order[:10]:
        p = pch_idx[i]
        print(f'{p:5d} {e_c4b[p]:10.1f} {e_other[p]:10.1f} {diff[p]:+10.1f}')

## 4. Noise Std comparison

Does noise variance differ across checkpoints? (Different weights → different ADC operating points)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (l1, l2) in zip(axes, pairs):
    s1 = stats[l1]['Std_mean']
    s2 = stats[l2]['Std_mean']
    mask = ~(np.isnan(s1) | np.isnan(s2))
    ax.scatter(s1[mask], s2[mask], s=8, alpha=0.5)
    lims = [0, max(np.nanmax(s1), np.nanmax(s2)) * 1.05]
    ax.plot(lims, lims, 'r--', lw=0.8, alpha=0.5)
    ax.set_xlabel(f'{l1} Std[noise]')
    ax.set_ylabel(f'{l2} Std[noise]')
    ax.set_title(f'{l1} vs {l2}')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

fig.suptitle('Per-pseudo_ch noise Std comparison', fontsize=13)
fig.tight_layout()
plt.show()

## 5. Summary statistics table

In [ ]:
print(f'{"":12s} {"Total obs":>12s} {"Coverage":>10s} {"Med E":>10s} {"Med |E|":>10s} {"Med Std":>10s}')
for label, t in tables.items():
    c = t['count']
    total = c.sum()
    coverage = (c.sum(axis=1) > 0).sum()  # bins with any data
    e = stats[label]['E_mean']
    s = stats[label]['Std_mean']
    valid = ~np.isnan(e)
    print(f'{label:12s} {total:12,} {coverage:8d}/{n_pch} '
          f'{np.nanmedian(e):+10.1f} {np.nanmedian(np.abs(e)):10.1f} {np.nanmedian(s):10.1f}')